# 02.2 — MobileNet + OpenCV DNN pe Raspberry Pi

Clasificare ImageNet și comparație cu măsurătorile DoReFaNet QNN/PYNQ deja obținute.
Selectați kernelul **QNN Raspberry Pi** și rulați celulele în ordine.
Nu este necesară reinstalarea mediului și nu se importă PYNQ/Caffe/TensorFlow.

**MobileNet v1 este alt model**, cu alte greutăți și preprocesare. Comparăm soluții pentru
aceeași sarcină, nu accelerarea aceluiași model. Diferența dintre imaginile celor două rulări
DoReFaNet nu împiedică comparația ordinului de mărime al timpilor pentru dimensiuni fixe.
Scorul unei predicții nu este acuratețea pe un set de test și nu se compară ca atare între modele.

La prima rulare se descarcă aproximativ 17 MB în `raspberry_pi/.deps/mobilenet`.
Sunt salvate numai fișierele modelului; nu se salvează capturi sau jurnale de rezultate.
Jupyter poate păstra rezultatele afișate când salvați notebookul.


In [ ]:
import gc, hashlib, platform, time, urllib.request
from pathlib import Path
import cv2
import numpy as np
from matplotlib import pyplot as plt
from IPython.display import display, Markdown
from threadpoolctl import threadpool_limits

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'raspberry_pi/qnn_cpu.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook inside QNN-MO-PYNQ/notebooks.')
MODEL_DIR = ROOT / 'raspberry_pi/.deps/mobilenet'
IMAGE_NAME = 'n01484850_0.jpg'
IMAGE_PATH = ROOT / 'notebooks/imagenet-samples' / IMAGE_NAME
THREAD_COUNTS = [1, 4]
WARMUP_RUNS = 2
MEASURED_RUNS = 5
BACKGROUND_LOAD = 'uStreamer / Home Assistant camera stream active'  # edit if needed
print('Machine:', platform.machine(), platform.platform())
print('Python:', platform.python_version(), 'OpenCV:', cv2.__version__, 'NumPy:', np.__version__)
print('Background:', BACKGROUND_LOAD)
if platform.machine() != 'aarch64':
    print('NOTE: this run is not on the target ARM64 Pi; timings are for this computer only.')

if not hasattr(cv2.dnn, 'readNetFromCaffe'):
    raise RuntimeError('This notebook needs OpenCV 4.x with the Caffe importer (tested with 4.10). Use the QNN Raspberry Pi kernel and Debian python3-opencv.')


## 1. Model verificat prin SHA256

Folosim MobileNet v1 din [shicai/MobileNet-Caffe](https://github.com/shicai/MobileNet-Caffe/tree/a0b0c46152e658559956158d7c2bf42bb029c09e).
Versiunea sursei este fixată; cele trei fișiere sunt verificate înainte de încărcare.
OpenCV citește formatul Caffe direct, fără instalarea frameworkului Caffe.


In [ ]:
MODEL_REVISION = 'a0b0c46152e658559956158d7c2bf42bb029c09e'
BASE_URL = f'https://raw.githubusercontent.com/shicai/MobileNet-Caffe/{MODEL_REVISION}/'
FILES = {
    'mobilenet_deploy.prototxt': '8e6a26b8f2c7cf7a066d571660cfd8a1544a5a25399df33ad499f3d733f16729',
    'mobilenet.caffemodel': '8d6edcd3dbd1356f2f19dd220c362c2ba8f44233a9b6c12ca6d0351cb0c446b6',
    'synset.txt': '946954b9717eb555b7319536f770f55e51bf255d37a99e9edf682022c596cfcf',
}
MODEL_DIR.mkdir(parents=True, exist_ok=True)
for name, expected_hash in FILES.items():
    path = MODEL_DIR / name
    if path.exists():
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected_hash:
            raise RuntimeError(f'Checksum mismatch: {path}. Remove this file and rerun the cell.')
    else:
        print('Downloading:', name, flush=True)
        partial = path.with_name(path.name + '.partial')
        try:
            digest = hashlib.sha256()
            with urllib.request.urlopen(BASE_URL + name, timeout=60) as response, partial.open('wb') as out:
                while True:
                    chunk = response.read(1024 * 1024)
                    if not chunk:
                        break
                    digest.update(chunk)
                    out.write(chunk)
            if digest.hexdigest() != expected_hash:
                raise RuntimeError('Download checksum mismatch: ' + name)
            partial.replace(path)
        finally:
            partial.unlink(missing_ok=True)
    print('Verified:', name)
labels = [line.strip().strip("'\"") for line in (MODEL_DIR / 'synset.txt').read_text().splitlines() if line.strip()]
if len(labels) != 1000 or any(not line.startswith('n') for line in labels):
    raise ValueError('Expected 1,000 ordered ImageNet synset labels.')


## 2. Imagine și preprocesare

Preprocesarea urmează exemplul `eval_image.py` al modelului: decupare pătrată centrală,
redimensionare la 224×224, ordine **BGR**, medii `[103.94, 116.78, 123.68]`, scalare `0.017`.
Redimensionarea se execută cu OpenCV. Nu aplicăm normalizarea DoReFaNet acestui model.


In [ ]:
def preprocess(path):
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(path)
    h, w = image.shape[:2]
    side = min(h, w)
    y, x = (h - side) // 2, (w - side) // 2
    crop = cv2.resize(image[y:y+side, x:x+side], (224, 224), interpolation=cv2.INTER_LINEAR)
    blob = cv2.dnn.blobFromImage(crop, scalefactor=0.017, size=(224, 224),
                                mean=(103.94, 116.78, 123.68), swapRB=False, crop=False)
    return image, blob

print('Input:', IMAGE_NAME)
print('Input SHA256:', hashlib.sha256(IMAGE_PATH.read_bytes()).hexdigest())
image, blob = preprocess(IMAGE_PATH)
fig, ax = plt.subplots(figsize=(7, 4))
ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
ax.axis('off')
plt.show()
plt.close(fig)


## 3. Inferență CPU — 1 și 4 threaduri

Backendul este explicit **OpenCV**, ținta **CPU**. `inference_s` include `setInput + forward`
pentru **întregul MobileNet**. `preprocess_s` include citirea fișierului, decuparea și blob-ul;
`postprocess_s` include validarea ieșirii și selectarea Top-5. Modelul are deja Softmax la ieșire.

Încărcarea modelului și prima inferență se raportează separat. Urmează două încălziri și
cinci rulări; tabelul raportează medianele. Afișarea, hash-urile și descărcarea nu sunt cronometrate.
Limita de threaduri OpenCV este solicitată, nu o garanție că fiecare operație folosește toate nucleele.


In [ ]:
def read_scores(output):
    scores = np.asarray(output).reshape(-1)
    if scores.shape != (1000,) or not np.all(np.isfinite(scores)):
        raise ValueError('Expected 1,000 finite scores.')
    if np.min(scores) < -1e-6 or not np.isclose(scores.sum(), 1.0, atol=1e-3):
        raise ValueError('Expected probabilities from the model Softmax layer.')
    return scores, np.argsort(scores)[-5:][::-1]

benchmarks = []
previous_threads = cv2.getNumThreads()
net = None
try:
    with threadpool_limits(limits=1):
        for threads in THREAD_COUNTS:
            cv2.setNumThreads(threads)
            t0 = time.perf_counter()
            net = cv2.dnn.readNetFromCaffe(str(MODEL_DIR / 'mobilenet_deploy.prototxt'),
                                         str(MODEL_DIR / 'mobilenet.caffemodel'))
            net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
            net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
            load_s = time.perf_counter() - t0
            t0 = time.perf_counter()
            net.setInput(blob)
            first_output = net.forward()
            first_s = time.perf_counter() - t0
            read_scores(first_output)
            for _ in range(WARMUP_RUNS):
                net.setInput(blob)
                net.forward()
            rows = []
            for _ in range(MEASURED_RUNS):
                t0 = time.perf_counter()
                _, measured_blob = preprocess(IMAGE_PATH)
                t1 = time.perf_counter()
                net.setInput(measured_blob)
                output = net.forward()
                t2 = time.perf_counter()
                scores, top5 = read_scores(output)
                t3 = time.perf_counter()
                rows.append([t1-t0, t2-t1, t3-t2, t3-t0])
            rows = np.asarray(rows)
            medians = np.median(rows, axis=0)
            benchmarks.append(dict(threads=threads, load_s=load_s, first_inference_s=first_s,
                                   preprocess_s=medians[0], inference_s=medians[1],
                                   postprocess_s=medians[2], pipeline_s=medians[3],
                                   min_inference_s=rows[:,1].min(), max_inference_s=rows[:,1].max(),
                                   scores=scores.copy(), top5=top5.copy(), samples_s=rows.copy()))
            print(f'{threads} thread(s): load {load_s:.4f} s; first inference {first_s:.4f} s; '
                  f'median inference {medians[1]:.4f} s; median pipeline {medians[3]:.4f} s')
            net = None
finally:
    net = None
    cv2.setNumThreads(previous_threads)


## 4. Rezultatul clasificării

In [ ]:
result = benchmarks[0]  # display the one-thread prediction
expected_synset = IMAGE_NAME.split('_')[0]
for index in result['top5']:
    print(f"{result['scores'][index]:8.2%}  {labels[index]}")
print('Top-1 matches filename:', labels[result['top5'][0]].split()[0] == expected_synset)
print('Expected class in Top-5:', any(labels[i].split()[0] == expected_synset for i in result['top5']))
for other in benchmarks[1:]:
    print(f"Max probability difference (1 vs {other['threads']} threads):",
          float(np.max(np.abs(result['scores'] - other['scores']))))
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh([labels[i].split(' ', 1)[1] for i in result['top5']], result['scores'][result['top5']])
ax.invert_yaxis()
ax.set_xlabel('Softmax score')
ax.set_title('MobileNet v1 — Top-5')
fig.tight_layout()
plt.show()
plt.close(fig)


## 5. Comparația cu măsurătorile QNN/PYNQ

Referințe din notebookurile executate de utilizator, 18 octombrie 2026:

| Soluție | Imagine | Top-1 | Scor |
|---|---|---|---:|
| RPi 4, DoReFaNet software | n01484850_0.jpg | great white shark | 98,23% |
| PYNQ, DoReFaNet CPU + FPGA | n02089973_2300.JPEG | English foxhound | 87,08% |

Ambele predicții au fost corecte pentru imaginea proprie. Acest tabel nu măsoară acuratețea
comparativă a modelelor. Rulările istorice sunt măsurători individuale; MobileNet are mediane după încălzire.

Pentru comparația calculului folosim cele **trei etape DoReFaNet** (prima convoluție + nucleul
cuantizat + FC) și **întregul `setInput + forward` MobileNet**. Toate exclud preprocesarea,
împachetarea externă și afișarea, însă modele și operații diferite sunt implicate.
Timpul FPGA de 0,057 s reprezintă numai o parte din DoReFaNet și nu este echivalent cu întregul MobileNet.


In [ ]:
# Historical measurements, not fresh measurements on this machine.
RPi_QNN = dict(first_cpu_s=0.3524, quantized_core_s=66.1156, last_cpu_s=0.1354)
PYNQ_QNN = dict(first_cpu_s=0.667, quantized_core_s=0.057, last_cpu_s=0.573)
RPI_COMPUTE_S = sum(RPi_QNN.values())
PYNQ_COMPUTE_S = 1.296  # total printed in the executed PYNQ notebook
# Rounded individual PYNQ stages sum to 1.297; preserve the reported total.
lines = ['| Solution | Compute time (s) | RPi QNN / solution | PYNQ QNN / solution |',
         '|---|---:|---:|---:|',
         f'| RPi 4 — DoReFaNet reference | {RPI_COMPUTE_S:.4f} | 1.00 | {PYNQ_COMPUTE_S/RPI_COMPUTE_S:.2f} |',
         f'| PYNQ — DoReFaNet CPU + FPGA | {PYNQ_COMPUTE_S:.4f} | {RPI_COMPUTE_S/PYNQ_COMPUTE_S:.2f} | 1.00 |']
for row in benchmarks:
    seconds = row['inference_s']
    lines.append(f"| MobileNet OpenCV — {row['threads']} thread(s) | {seconds:.4f} | "
                 f"{RPI_COMPUTE_S/seconds:.2f} | {PYNQ_COMPUTE_S/seconds:.2f} |")
display(Markdown('\n'.join(lines)))
print('Ratio > 1 means the solution in that row is faster than the reference column.')

names = ['RPi QNN reference', 'PYNQ CPU + FPGA'] + [f"MobileNet {r['threads']} thread(s)" for r in benchmarks]
values = [RPI_COMPUTE_S, PYNQ_COMPUTE_S] + [r['inference_s'] for r in benchmarks]
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(names, values, color=['gray', 'darkorange'] + ['steelblue'] * len(benchmarks))
ax.set_xscale('log')
ax.set_xlabel('Compute time, seconds — logarithmic scale; lower is faster')
ax.invert_yaxis()
for y, value in enumerate(values):
    ax.text(value * 1.08, y, f'{value:.4f} s', va='center')
ax.set_xlim(min(values) / 2, max(values) * 3)
fig.tight_layout()
plt.show()
plt.close(fig)

lines = ['| OpenCV threads | Preprocess (s) | Inference median [min, max] (s) | Postprocess (s) | Pipeline median (s) |',
         '|---:|---:|---:|---:|---:|']
for r in benchmarks:
    lines.append(f"| {r['threads']} | {r['preprocess_s']:.4f} | {r['inference_s']:.4f} "
                 f"[{r['min_inference_s']:.4f}, {r['max_inference_s']:.4f}] | "
                 f"{r['postprocess_s']:.4f} | {r['pipeline_s']:.4f} |")
display(Markdown('\n'.join(lines)))
print('The historical RPi QNN pipeline was 67.0889 s; no equivalent full PYNQ pipeline was measured.')
print('Results remain in notebook memory. No result files were written.')


## Interpretare

- Comparați întâi timpul întregului calcul: MobileNet `inference_s` vs 1,296 s PYNQ și 66,6034 s QNN software.
- Raporturile descriu aceste implementări și rulări, nu superioritatea universală a CPU sau FPGA.
- Îmbunătățirea față de QNN software arată de ce referința C++/HLS nu este o măsură a performanței maxime CPU.
- Pentru concluzii despre acuratețe sunt necesare multe imagini etichetate comune, nu scorurile a două imagini.
- Păstrați aceeași alimentare/răcire și aceeași încărcare de fundal; nu rulați simultan notebookurile QNN.

Surse: [model, greutăți și etichete](https://github.com/shicai/MobileNet-Caffe/tree/a0b0c46152e658559956158d7c2bf42bb029c09e),
[preprocesarea autorului](https://github.com/shicai/MobileNet-Caffe/blob/a0b0c46152e658559956158d7c2bf42bb029c09e/eval_image.py),
[OpenCV DNN](https://docs.opencv.org/4.x/d6/d0f/group__dnn.html).
